> ### **Building with Sarvam**
>
> **An open teaching kit for the Sarvam AI stack.**
>
> Notebook maintained by **Dr. Bhaveshkumar C. Dharmani** — Founder & AI Mentor, AIVidhya4Sarvam.
PhD (ICT), DA-IICT Gandhinagar · https://www.aividhya.in/ · https://www.aividhya4sarvam.in/ · bhavesh@aividhya.in · https://www.linkedin.com/in/bhaveshdharmani/
>
> Drafted with AI assistance and stress-tested cell by cell in live workshop sessions. Runs against your own Sarvam API key from dashboard.sarvam.ai, with a live ₹ cost meter after every call.
>
> Apache 2.0 · Issues and PRs welcome at github.com/dharmanibc/building-with-sarvam.

<div style="background:#12172E;color:#fff;padding:20px 24px;border-radius:8px">
<div style="color:#FF8A3D;font-size:12px;letter-spacing:2px;font-weight:700">LAB 03 · SPEECH OUT</div>
<div style="font-size:26px;font-weight:700;margin-top:6px">Bulbul — voices, controls, streaming, and 60% of your bill</div>
<div style="color:#FFB37A;font-size:14px;margin-top:8px">Personas · pitch/pace/loudness · REST vs stream vs WebSocket · TTFB · telephony formats</div>
</div>

**Time:** 50 min &nbsp;·&nbsp; **Est. cost:** ≈ ₹6 &nbsp;·&nbsp; **Prereq:** Lab 00

## Why this lab matters more than it looks

In a voice product, **text-to-speech is roughly 60% of your per-call cost** — far more
than the LLM everyone obsesses over. It is also the component that decides whether
your agent sounds like a person or a train announcement.

In [1]:
# ── Standard lab header. Run this first in every notebook. ─────────────────
import os, sys, json, time, math, wave, io
from pathlib import Path

# pip install sarvamai python-dotenv
from sarvamai import SarvamAI

# Put SARVAM_API_KEY in a .env file next to this notebook, or set it here.
# try:
#     from dotenv import load_dotenv; load_dotenv()
# except ImportError:
#     pass

from dotenv import load_dotenv, find_dotenv
# Finds your key without hardcoding anyone's filesystem. Tried in order:
#   1. SARVAM_API_KEY already set in the environment
#   2. the file named by SARVAM_ENV_FILE, if you set that variable
#   3. a .env beside this notebook, or in any parent folder
load_dotenv(os.environ.get("SARVAM_ENV_FILE") or find_dotenv(usecwd=True))

API_KEY = os.environ.get("SARVAM_API_KEY")
assert API_KEY, (
    "SARVAM_API_KEY not found.\\n"
    "Create a .env next to this notebook containing:  SARVAM_API_KEY=sk_...\\n"
    "or point SARVAM_ENV_FILE at an existing env file.\\n"
    "Free key + Rs 1000 credit: https://indus.sarvam.ai/"
)

client = SarvamAI(api_subscription_key=API_KEY)
DATA = Path("./data"); DATA.mkdir(exist_ok=True)
OUT  = Path("./out");  OUT.mkdir(exist_ok=True)
print("SDK ready ·", sys.version.split()[0])

SDK ready · 3.13.9


In [2]:
# ── The ₹ meter. Every lab uses this. ──────────────────────────────────────
# Rates as of August 2026. Verify at docs.sarvam.ai/api/getting-started/pricing
RATES = {
    "stt_per_hour":            30.00,
    "stt_diarized_per_hour":   45.00,
    "translate_per_10k":       20.00,
    "transliterate_per_10k":   20.00,
    "lid_per_10k":              3.50,
    "tts_v2_per_10k":          15.00,
    "tts_v3_per_10k":          30.00,
    "llm_in_per_1m":           29.28,
    "llm_cached_in_per_1m":    10.98,
    "llm_out_per_1m":          73.20,
    "doc_per_page":             0.50,
    "samvaad_per_min":          3.50,
}

class CostMeter:
    """Running ₹ tally for Sarvam API calls, calculated from published rates
    as of August 2026 (see RATES dict). Estimate only — actual invoice may
    differ due to rate changes, rounding, prompt caching and free-tier
    consumption. In fact, the meter is a conservative upper bound on cost 
    in almost every real scenario if rate changes are not applied. Verify 
    current pricing at docs.sarvam.ai/api/getting-started/pricing.
    """
    def __init__(self): self.items = []

    def add(self, label, rupees, detail=""):
        self.items.append({"label": label, "inr": rupees, "detail": detail})
        return rupees

    # --- convenience wrappers, one per billing unit -----------------------
    def stt(self, seconds, diarized=False):
        r = RATES["stt_diarized_per_hour" if diarized else "stt_per_hour"] / 3600 * seconds
        return self.add("STT" + (" +diar" if diarized else ""), r, f"{seconds:.1f}s")

    def tts(self, chars, v3=True):
        r = RATES["tts_v3_per_10k" if v3 else "tts_v2_per_10k"] / 10_000 * chars
        return self.add(f"TTS {'v3' if v3 else 'v2'}", r, f"{chars} chars")

    def text(self, chars, kind="translate"):
        r = RATES[f"{kind}_per_10k"] / 10_000 * chars
        return self.add(kind, r, f"{chars} chars")

    def llm(self, in_tok, out_tok, cached_tok=0):
        r = ((in_tok - cached_tok) * RATES["llm_in_per_1m"]
             + cached_tok * RATES["llm_cached_in_per_1m"]
             + out_tok * RATES["llm_out_per_1m"]) / 1_000_000
        return self.add("LLM", r, f"{in_tok} in / {out_tok} out / {cached_tok} cached")

    def doc(self, pages):
        return self.add("DocAI", RATES["doc_per_page"] * pages, f"{pages} pages")

    def report(self):
        if not self.items:
            print("nothing billed yet"); return 0.0
        w = max(len(i["label"]) for i in self.items) + 2
        print("─" * (w + 34))
        for i in self.items:
            print(f"{i['label']:<{w}} ₹{i['inr']:>9.4f}   {i['detail']}")
        total = sum(i["inr"] for i in self.items)
        print("─" * (w + 34))
        print(f"{'TOTAL':<{w}} ₹{total:>9.4f}")
        print(f"{'':<{w}}  (₹100 free credit → ₹{100-total:.2f} left)")
        return total

cost = CostMeter()
print("cost meter armed")

cost meter armed


### 1 · The voice catalogue

In [5]:
# SPEAKERS = ["anushka", "abhilash", "manisha", "vidya", "arya", "karun", "hitesh"] --> were for Bulbul:v2 
"""Available speakers for bulbul:v3 are:aditya, ritu, ashutosh, priya, neha, rahul, pooja, rohan, simran, kavya, amit, dev, ishita, 
shreya, ratan, varun, manan, sumit, roopa, kabir, aayan, shubh, advait, anand, tanya, tarun, sunny, mani, gokul, vijay, shruti, suhani,
mohit, kavitha, rehan, soham, rupali, niharika."""


SPEAKERS = ["aditya", "ritu", "ashutosh", "priya", "neha", "rahul", "pooja"]

LINE = "आपकी किस्त पंद्रह तारीख को देय है। कृपया समय पर भुगतान करें।"

from sarvamai.play import save
from IPython.display import Audio, display

for spk in SPEAKERS[:4]:                     # trim to conserve credit
    try:
        a = client.text_to_speech.convert(
            text=LINE, language_code="hi-IN", model="bulbul:v3", speaker=spk)
        p = OUT / f"voice_{spk}.wav"; save(a, str(p)); cost.tts(len(LINE), v3=False)
        print(spk); display(Audio(str(p)))
    except Exception as e:
        print(f"{spk}: unavailable ({e})")

aditya


ritu


ashutosh


priya


**Pick by persona, not preference.**

| Persona | Use for | Wrong for |
|---|---|---|
| Conversational / Friendly | Support, assistants, reminders | Regulatory notices |
| News / Authoritative | Announcements, compliance readouts | A chatty helper |
| Entertainment / Dynamic | Edtech, media, content | A bank |
| Consistent / Neutral | Long-form narration | Anything needing warmth |

### 2 · The control surface

In [7]:
# ⚠️ pitch and loudness are bulbul:v2-only controls — bulbul:v3 rejects them
# outright (400: "Pitch and loudness parameters are currently not supported
# for the Bulbul V3 model"). pace is the only numeric control surface left.

VARIANTS = [
    ("baseline",   dict()),
    ("slow",       dict(pace=0.75)),
    ("fast",       dict(pace=1.3)),
    # ("low_pitch",  dict(pitch=-0.3)),     # 
    # ("high_pitch", dict(pitch=0.3)),
    # ("loud",       dict(loudness=1.4)),
]

for name, kw in VARIANTS:
    a = client.text_to_speech.convert(
        text=LINE, language_code="hi-IN",
        model="bulbul:v3", speaker="pooja", **kw)
    p = OUT / f"ctrl_{name}.wav"; save(a, str(p)); cost.tts(len(LINE), v3=False)
    print(f"{name:<11}", kw); display(Audio(str(p)))

baseline    {}


slow        {'pace': 0.75}


fast        {'pace': 1.3}


| Parameter | Range | Notes |
|---|---|---|
| `pitch` | roughly −1 … 1 | Small moves. ±0.2 is already noticeable, Only for Bulbul:v2 |
| `pace` | roughly 0.5 … 2 | 0.9 often sounds more natural than 1.0 for Indic |
| `loudness` | roughly 0.3 … 3 | Normalise instead where you can,  Only for Bulbul:v2 |
| `speech_sample_rate` | 8000 / 16000 / 22050 / 24000 | **8000 for telephony** |
| `enable_preprocessing` | bool | Expands numbers, dates, currency. Usually leave on |
| `output_audio_codec` | mp3, wav, linear16, **mulaw**, alaw, opus, flac, aac | mulaw/alaw = phone |

### 3 · Telephony output — the format that breaks people

In [8]:
# Phone systems want 8 kHz mu-law. Get this wrong and you hear static or chipmunks.
tel = client.text_to_speech.convert(
    text=LINE,
    language_code="hi-IN",
    model="bulbul:v2",
    speaker="anushka",
    speech_sample_rate=8000,
    output_audio_codec="mulaw",     # <- what Twilio/Exotel expect
)
save(tel, str(OUT / "telephony.raw")); cost.tts(len(LINE), v3=False)
print("wrote telephony.raw — 8 kHz mu-law, ready for a phone bridge")

wrote telephony.raw — 8 kHz mu-law, ready for a phone bridge


---
## 4 · Three delivery paths, and the number that matters

For a conversation, **time to first audio byte** is the whole user experience.
Measure it yourself — your numbers beat the docs.

In [9]:
import time

LONG = ("आपका ऋण आवेदन स्वीकृत हो गया है। कृपया अगले चरण के लिए दस्तावेज़ जमा करें। "
        "किसी भी सहायता के लिए हमारी ग्राहक सेवा से संपर्क करें।")

# --- Path 1: REST (batch) — nothing until the whole thing is generated
t0 = time.perf_counter()
a = client.text_to_speech.convert(text=LONG, language_code="hi-IN",
                                  model="bulbul:v2", speaker="anushka")
rest_total = time.perf_counter() - t0
cost.tts(len(LONG), v3=False)
print(f"REST      first audio = {rest_total*1000:>7.0f} ms   (== total)")

REST      first audio =     624 ms   (== total)


In [10]:
# --- Path 2: HTTP streaming — audio starts arriving progressively
t0 = time.perf_counter(); first = None; nbytes = 0
stream = client.text_to_speech.convert_stream(
    text=LONG, language_code="hi-IN", model="bulbul:v2", speaker="anushka")
for chunk in stream:
    if first is None:
        first = time.perf_counter() - t0
    nbytes += len(chunk) if isinstance(chunk, (bytes, bytearray)) else 0
http_total = time.perf_counter() - t0
cost.tts(len(LONG), v3=False)
print(f"HTTP str  first audio = {first*1000:>7.0f} ms   total = {http_total*1000:.0f} ms")

HTTP str  first audio =     696 ms   total = 1078 ms


In [11]:
# --- Path 3: WebSocket — lowest latency, needs an explicit end signal
# connect() only accepts model/send_completion_event — language_code, speaker
# etc. are set via ws.configure() AFTER connecting, not as connect() kwargs.
# connect() is also an async generator only on AsyncSarvamAI, not the sync client.
import base64
from sarvamai import AsyncSarvamAI

async_client = AsyncSarvamAI(api_subscription_key=API_KEY)

async def ws_tts(text):
    t0 = time.perf_counter(); first = None; buf = bytearray()
    async with async_client.text_to_speech_streaming.connect(model="bulbul:v2") as ws:
        await ws.configure(target_language_code="hi-IN", speaker="anushka")
        await ws.convert(text)
        await ws.flush()
        async for m in ws:
            if getattr(m, "type", "") == "audio":
                if first is None: first = time.perf_counter() - t0
                buf.extend(base64.b64decode(m.data.audio))   # audio arrives base64-encoded
    return first, time.perf_counter() - t0, len(buf)

first_ws, total_ws, n = await ws_tts(LONG)
cost.tts(len(LONG), v3=False)
print(f"WebSocket first audio = {first_ws*1000:>7.0f} ms   total = {total_ws*1000:.0f} ms")


WebSocket first audio =     940 ms   total = 60509 ms


> **The latency budget.** A human conversation tolerates roughly **800 ms** of silence
> before it feels broken. Your budget: VAD endpointing + STT finalisation + LLM first
> token + **TTS first byte** + network. One non-streaming hop blows the whole thing —
> which is why the REST number above disqualifies it for live agents.

---
## 5 · The version choice that decides your margin

In [12]:
CHARS_PER_CALL = 1600          # ~1.8 min of agent speech
for volume in [1_000, 10_000, 100_000, 1_000_000]:
    v2 = volume * CHARS_PER_CALL / 10_000 * 15
    v3 = volume * CHARS_PER_CALL / 10_000 * 30
    print(f"{volume:>9,} calls/mo   v2 ₹{v2:>12,.0f}   v3 ₹{v3:>12,.0f}   Δ ₹{v3-v2:>12,.0f}")

    1,000 calls/mo   v2 ₹       2,400   v3 ₹       4,800   Δ ₹       2,400
   10,000 calls/mo   v2 ₹      24,000   v3 ₹      48,000   Δ ₹      24,000
  100,000 calls/mo   v2 ₹     240,000   v3 ₹     480,000   Δ ₹     240,000
1,000,000 calls/mo   v2 ₹   2,400,000   v3 ₹   4,800,000   Δ ₹   2,400,000


In [13]:
# Hear the difference yourself before you decide it doesn't matter
# for model in ["bulbul:v2", "bulbul:v3"]:
# a = client.text_to_speech.convert(text=LONG, language_code="hi-IN",
#                                   model="bulbul:v2", speaker="anushka")

model="bulbul:v2"
a = client.text_to_speech.convert(text=LINE, language_code="hi-IN",
                                  model=model, speaker="anushka")
p = OUT / f"cmp_{model.replace(':','_')}.wav"; save(a, str(p))

cost.tts(len(LINE), v3=(model == "bulbul:v2"))
print(model); display(Audio(str(p)))

model="bulbul:v3"
b = client.text_to_speech.convert(text=LINE, language_code="hi-IN",
                                  model=model, speaker="shubh")
p = OUT / f"cmp_{model.replace(':','_')}.wav"; save(b, str(p))

cost.tts(len(LINE), v3=(model == "bulbul:v3"))
print(model); display(Audio(str(p)))

bulbul:v2


bulbul:v3


**The decision is a product decision, not a technical one.** For an outbound reminder
IVR, v2 is almost certainly fine and halves your largest cost line. For a premium
concierge assistant, v3 earns its price. Decide deliberately — most teams never do.

### 6 · Long text — sentence splitting and buffering

In [14]:
ESSAY = (LONG + " ") * 6      # ~1,000 characters

a = client.text_to_speech.convert(
    text=ESSAY,
    language_code="hi-IN",
    model="bulbul:v2",
    speaker="anushka",
    enable_preprocessing=True,     # expands ₹, dates, numbers
)
save(a, str(OUT / "long.wav")); cost.tts(len(ESSAY), v3=False)
print(f"{len(ESSAY)} chars → {OUT/'long.wav'}")
display(Audio(str(OUT / "long.wav")))

786 chars → out/long.wav


In [15]:
cost.report()

──────────────────────────────────────────
TTS v2   ₹   0.0900   60 chars
TTS v2   ₹   0.0900   60 chars
TTS v2   ₹   0.0900   60 chars
TTS v2   ₹   0.0900   60 chars
TTS v2   ₹   0.0900   60 chars
TTS v2   ₹   0.0900   60 chars
TTS v2   ₹   0.0900   60 chars
TTS v2   ₹   0.0900   60 chars
TTS v2   ₹   0.0900   60 chars
TTS v2   ₹   0.0900   60 chars
TTS v2   ₹   0.0900   60 chars
TTS v2   ₹   0.1950   130 chars
TTS v2   ₹   0.1950   130 chars
TTS v2   ₹   0.1950   130 chars
TTS v3   ₹   0.1800   60 chars
TTS v3   ₹   0.1800   60 chars
TTS v2   ₹   1.1790   786 chars
──────────────────────────────────────────
TOTAL    ₹   3.1140
          (₹100 free credit → ₹96.89 left)


3.114

---
## ✅ Checkpoint

- [ ] You listened to ≥4 voices and can name which persona fits your product
- [ ] You have **your own** TTFB numbers for REST / HTTP / WebSocket
- [ ] You produced 8 kHz mu-law output for a phone bridge
- [ ] You can state, in rupees, what v2-vs-v3 costs at your volume

## 🧪 Try this

1. Read your agent's actual script aloud, then shorten it 20%. Recompute the TTS cost.
   **Copywriting is cost control.**
2. Generate the same line in all 10 Indic languages and play them back to back.
3. Build a pronunciation dictionary for 5 terms in your domain and A/B it with a native speaker.